In [9]:
import pygame
from classes.logic import Card as card
from classes.logic import Player as player
from classes.logic import Game as game
from classes.logic import Deck as deck
from functions import *
from assets import *
import random
import copy
import time
import pandas as pd
import traceback

In [2]:
pt = ["AI random", "AI playrandom", "AI greedy", "AI rule", "AI mcts short", "AI mcts long"]

In [3]:
players = [pt[1], pt[2], pt[3], pt[4], pt[5]]

In [10]:
def play(hpp = None):
    try:
        g = game.Game(players, hpp)
        for i in range(4):
            g.season = random.sample(g.seasondeck, 1)[0]
            g.seasondeck.remove(g.season)
            for p in g.players:
                p.season_buff = g.s_ref[g.season]['akce']
            g.hands = []
            for i in range(len(g.players)):
                new_hand = deck.PlayerDeck(i, g.game_deck, g.reference)
                for j in new_hand.cardids:
                    g.game_deck.remove(j)
                g.hands.append(new_hand)
            for j in range(8):
                for p in range(len(g.players)):
                        person = g.players[p]
                        g.current_player = p
                        g.current_deck = (g.current_player+g.firstplayerdeck)%len(g.players)
                        d = g.hands[g.current_deck]
                        person.start_turn(d, g)
                        info = person.choose(g)
                        while info[0]!="end_turn":
                            id = info[2]
                            if info[0]=="store_hand":
                                    d.store_card(id, person, g)
                            else:
                                if info[0]=="play_stored":
                                    d = person.stored
                                karta = d.cards[id]
                                if karta.card_type == "monster" and karta.cards>0:
                                        deck.sample_cards(g, person, karta.cards)
                                msg = d.play_card(id, person, g)
                                if msg == 'fialova':
                                    person.play_purple(karta)
                                if msg == 'biom':
                                    if len(info)>1:
                                        person.play_biom_e(karta, info[2])
                            d = g.hands[g.current_deck]
                            info = person.choose(g)
                        person.had_played = False
                g.turn += 1
                g.firstplayerdeck = (g.firstplayerdeck+len(g.hands)+(-1)**g.round)%len(g.players)
                g.current_deck = (g.current_player+g.firstplayerdeck)%len(g.players)
            dice = random.sample([0,0,0,0,1,2], 1)[0] + random.sample([0,0,0,0,1,2], 1)[0]
            goals = []
            for play in range(len(g.players)):
                    vals = g.players[play].end_season(dice,  g.s_ref[g.season]['akce'])
                    goals.append(vals[5])
            if goals.count(max(goals))==1:
                    index = goals.index(max(goals))
                    g.players[index].seasons_won +=1
            for person in g.players:
                        person.monsters.cards.extend(person.played.cards)
                        person.played.cards = []
                        person.occupied ={"modra":[], "cerna":[], "hneda":[], "zelena":[], "zlata":[], "fialova":[]}
                        for mct in g.mcts:
                            mcts = g.players[mct]
                            mcts.seen = [-1]*len(g.players)
                        for biom in person.bioms:
                            person.bioms[biom][0] += person.bioms[biom][1]
                            person.bioms[biom][1] = 0
            g.round +=1
            g.turn = 0
            
        g.end_game()
        return "OK", [g.order, [sum(result) for result in g.results]]
    except Exception as e:
        return traceback.format_exc(), copy.deepcopy(g)


In [11]:
play()

('OK', [[2, 4, 3, 0, 1], [33, -26, 74, 49, 61]])

In [ ]:
results = []
fails = []
messages = []
sim = 0
while sim < 250:
    msg, res = play()
    if msg == "OK":
        results.append(res)
        sim += 1
    else:
        fails.append(res)
        messages.append(msg)
out = pd.DataFrame(results, columns=["won", "points"])
out.to_csv("long2.csv", index=False)


KeyboardInterrupt: 

In [100]:
len(fails)

106

In [13]:
t = pd.read_csv("out.csv")
t

,Unnamed: 0,HP,Resource
0,0,1,83


In [39]:
fails2 =fails

In [41]:
len(fails2)

131

In [75]:
fails1[0].players[0].known[3][0][0]

['PO_11', 'U_05', 'ZO_14', 'TO_06', 'TU_11']

In [67]:
for c in fails1[0].players[1].stored.cards:
    print(c.id)

TU_05
JO_13
ZO_09
TU_02
JO_14
ZO_03
TU_21
ZO_04
ZO_01
TO_14
K_17


In [68]:
fails1[0].players[0].sets

[[],
 [(0, 0, 0),
  0,
  (0, 1, 1),
  (0, 0, 2),
  0,
  (1, 0, 1),
  (2, 1, 0),
  (2, 0, 2),
  0,
  0,
  (3, 2, 0)],
 [(0, 1, 1), (1, 1, 2), 0, (2, 1, 1), 0, (3, 2, 0), (3, 0, 0)]]

In [83]:
for i in range(len(fails1)):
    for j in range(len(fails1[i].players[0].sets)):
        for k in range(len(fails1[i].players[0].sets[j])):
            if fails1[i].players[0].sets[j][k] == 0:
                break
            coords = fails1[i].players[0].sets[j][k]
            if fails1[i].players[0].known[coords[0]][coords[1]][coords[2]]==[]:
                print(fails1[i].players[j].stored.cards[k].id)

K_05
K_17
K_21
K_13
K_01
K_17
K_05
K_01
K_09
K_05
K_05
K_13
K_01
K_13
K_05
K_17
K_01
K_17
K_13
K_05
K_13
K_13
K_05
K_17
K_05


In [84]:
for i in range(len(fails2)):
    for j in range(len(fails2[i].players[0].sets)):
        for k in range(len(fails2[i].players[0].sets[j])):
            if fails2[i].players[0].sets[j][k] == 0:
                break
            coords = fails2[i].players[0].sets[j][k]
            if fails2[i].players[0].known[coords[0]][coords[1]][coords[2]]==[]:
                print(fails2[i].players[j].stored.cards[k].id)

K_05
K_17
K_13
K_05
K_17
K_17
K_13
K_21
K_01
K_05
K_13
K_17


In [11]:
test = [[1, 6], [0, 250], [0, 100]]
test = pd.DataFrame(test, columns=["HP", "Resource"])
test.to_csv("test.csv")

In [ ]:
g = game.Game(players, hps[0])
for i in range(4):
    g.season = random.sample(g.seasondeck, 1)[0]
    g.seasondeck.remove(g.season)
    for p in g.players:
        p.season_buff = g.s_ref[g.season]['akce']
    g.hands = []
    for i in range(len(g.players)):
        new_hand = deck.PlayerDeck(i, g.game_deck, g.reference)
        for j in new_hand.cardids:
            g.game_deck.remove(j)
        g.hands.append(new_hand)
    for j in range(8):
        for p in range(len(g.players)):
                person = g.players[p]
                g.current_player = p
                g.current_deck = (g.current_player+g.firstplayerdeck)%len(g.players)
                d = g.hands[g.current_deck]
                person.start_turn(d, g)
                start_time = time.time()
                info = person.choose(g)
                print(time.time() - start_time)
                while info[0]!="end_turn":
                    id = info[2]
                    if info[0]=="store_hand":
                            d.store_card(id, person, g)
                    else:
                        if info[0]=="play_stored":
                            d = person.stored
                        karta = d.cards[id]
                        if karta.card_type == "monster" and karta.cards>0:
                                deck.sample_cards(g, person, karta.cards)
                        msg = d.play_card(id, person, g)
                        if msg == 'fialova':
                            person.play_purple(karta)
                        if msg == 'biom':
                            if len(info)>1:
                                person.play_biom_e(karta, info[2])
                    d = g.hands[g.current_deck]
                    info = person.choose(g)
                person.had_played = False
        g.turn += 1
        g.firstplayerdeck = (g.firstplayerdeck+len(g.hands)+(-1)**g.round)%len(g.players)
        g.current_deck = (g.current_player+g.firstplayerdeck)%len(g.players)
    dice = random.sample([0,0,0,0,1,2], 1)[0] + random.sample([0,0,0,0,1,2], 1)[0]
    goals = []
    for play in range(len(g.players)):
            vals = g.players[play].end_season(dice,  g.s_ref[g.season]['akce'])
            goals.append(vals[5])
    if goals.count(max(goals))==1:
            index = goals.index(max(goals))
            g.players[index].seasons_won +=1
    for person in g.players:
                person.monsters.cards.extend(person.played.cards)
                person.played.cards = []
                person.occupied ={"modra":[], "cerna":[], "hneda":[], "zelena":[], "zlata":[], "fialova":[]}
                for mct in g.mcts:
                    mcts = g.players[mct]
                    mcts.seen = [-1]*len(g.players)
                for biom in person.bioms:
                    person.bioms[biom][0] += person.bioms[biom][1]
                    person.bioms[biom][1] = 0
    g.round +=1
    
g.end_game()
print(g.order)



In [ ]:
g.results

In [ ]:
a =[1,2,2]

In [ ]:
if type(a) == list:
    print("list")